In [1]:
import pandas as pd
import numpy as np
import time
import kagglehub
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.feature_selection import SequentialFeatureSelector

**캐글 pistachio 데이터**를 이용하여

고차원 데이터에서 차원의 저주현상이 발생함을 확인하고, 이를 다양한 차원축소 기법을 이용하여 해결하는 과정입니다.

In [2]:
#1. 데이터셋 불러오기
#만약 구글코랩을 사용하신다면, 아래 코드를 각자의 저장 경로에 맞춰 수정한 뒤 불러와주시고
#주피터나 파이썬 등을 이용하시는 분들이라면 구글코랩 경로 삭제 후 각 프로그램에 맞는 코드 사용하시면 됩니다

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import pandas as pd
file_path='/content/drive/MyDrive/Pistachio.xlsx'
df=pd.read_excel(file_path)
print(df.head())

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Pistachio.xlsx'

In [ ]:
#2. 데이터 구조확인해보기
print(df. columns.tolist)

28개의 feature와 1개의 타겟 (class)로 이루어진 데이터

feature는 모두 수치형, class는 범주형

이 데이터의 최종 목적은 "분류" 입니다.


기하학적 특징: Area, Perimeter, Major_Axis, Minor_Axis 등


색상 특징: Mean_RR, Mean_RG, Mean_RB


통계적 특징: StdDev, Skew, Kurtosis

우선, 이러한 원본 데이터를 그대로 분석에 사용할 경우 어떤 문제가 발생하는지 파악하도록 하겠습니다.

In [ ]:
#3. 원본 데이터 그대로 사용 시 발생하는 문제 파악
X = df.drop('Class', axis=1)
y = df['Class']
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled = scaler.transform(X_test_raw)

Q. 스케일링을 진행해야하는 이유는? (현재 데이터가 고차원 데이터임을 전제하고, 데이터를 분석해야 한다고 생각해주세요.)

데이터의 단위가 모두 다르기 때문에

In [ ]:
model_orig = RandomForestClassifier(n_estimators=100, max_depth=20, random_state=42) #분류모델 사용, 하이퍼파라미터 찾는 과정 생략)
model_orig.fit(X_train_scaled, y_train)
end_time = time.time()

train_acc_orig = accuracy_score(y_train, model_orig.predict(X_train_scaled))
test_acc_orig = accuracy_score(y_test, model_orig.predict(X_test_scaled))

print(f"훈련 정확도: {train_acc_orig:.4f}")
print(f"테스트 정확도: {test_acc_orig:.4f}")
print(f"새로운 격차: {train_acc_orig - test_acc_orig:.4f}")

Q. 해당 결과의 문제점은? (*고차원 데이터의 차원의 저주 문제를 생각)

=> 과적합 발생, 훈련 데이터의 정확도에 비해 테스트 데이터의 정확도가 떨어진다


1. 특징선택 (필요한 변수만 남기기)

1).  filter method= thresold를 사전에 지정해야하는 방법





In [ ]:
X = df.drop('Class', axis=1)
y = df['Class']

le = LabelEncoder()
y_encoded = le.fit_transform(y)

# 데이터 스케일링 (분산 기반 및 카이제곱 검정에 필수)
scaler = MinMaxScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

In [ ]:
#1. 분산기반

selector_var = VarianceThreshold(threshold=0.01) #Q.적절한 함수 작성
X_var = selector_var.fit_transform(X_scaled)
print("분산 기반 선택된 변수 개수:", X_var.shape[1])

In [ ]:
#2. 상관계수
# X와 인코딩된 정답을 합쳐서 상관계수 계산
df_corr = X.copy()
df_corr['Target'] = y_encoded
top_corr = df_corr.corr()['Target'].abs().sort_values(ascending=False)
print("정답과 상관관계가 높은 변수 TOP 5:\n", top_corr.head(6)[1:]) #사전에 정한 threshold

In [ ]:
#3. 카이제곱검정

selector_chi2 = SelectKBest(score_func=chi2, k=10) #Q.적절한 함수 작성
selector_chi2.fit(X_scaled, y_encoded)
selected_features = X.columns[selector_chi2.get_support()]
print("카이제곱 검정으로 선택된 10개 변수:\n", list(selected_features))

2). wrapper method

In [ ]:
model = RandomForestClassifier(n_estimators=10, random_state=42)

# 1. 전진 선택법 (Forward)
sfs_forward = SequentialFeatureSelector(model, n_features_to_select=5, direction='forward')#Q. 빈칸 채우기
sfs_forward.fit(X, y)
print("전진 선택된 변수:", list(X.columns[sfs_forward.get_support()]))

# 2. 후진 제거법 (Backward)
sfs_backward = SequentialFeatureSelector(model, n_features_to_select=5, direction='backward') #Q. 빈칸 채우기
sfs_backward.fit(X, y)
print("후진 제거 후 남은 변수:", list(X.columns[sfs_backward.get_support()]))

Q. filter method과 wrapper method의 차이점은 무엇인가요? 변수를 찾아내는 과정에서 무엇을 사용하였는지에 초점을 맞춰 답변부탁드립니다 .

사전의 threshold 유무 차이

Filter method는 통계적 측정으로 각 변수를 독립적으로 평가하여 선택하고, wrapper method는 머신러닝 모델을 사용하여 변수 조합의 성능을 평가하며 반복적으로 최적의 변수를 찾아냅니다.

2. 특징추출

특징 추출의 대표적인 방법인 pca 실습코드입니다.
우선 전체 변수를 사용하여서, scree plot을 이용해 적절한 주성분의 개수를 찾습니다.

(하이퍼파라미터 결정)

이후 분산설명력을 확인합니다.

In [ ]:
pca_full = PCA()
pca_full.fit(X_scaled)

exp_var_ratio = pca_full.explained_variance_ratio_
cum_var_ratio = np.cumsum(exp_var_ratio)

plt.figure(figsize=(10, 6))

plt.bar(range(1, len(exp_var_ratio) + 1), exp_var_ratio, alpha=0.5, align='center', label='Individual explained variance')

plt.step(range(1, len(cum_var_ratio) + 1), cum_var_ratio, where='mid', label='Cumulative explained variance', color='red')

plt.axhline(y=0.8, color='black', linestyle='--')
plt.text(20, 0.82, '80% Threshold', color = 'black', fontsize=12)

plt.ylabel('Explained variance ratio')
plt.xlabel('Principal component index')
plt.title('Scree Plot for Pistachio Dataset')
plt.legend(loc='best')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=5) #Q.scree plot을 보고 빈칸 채우기
pc_results = pca.fit_transform(X_scaled)

# --- 결과 출력 ---
print(f"PCA 설명력(분산 보존율): {np.sum(pca.explained_variance_ratio_)*100:.2f}%")
